# Erlang C Calculator — API Tests

Place this notebook in `tests`, select the `Erlang C (.venv)` kernel, run Setup first, then run API Tests 1–15 in order. Uvicorn does not need to be running.

## Setup 

In [28]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
from IPython.display import Markdown, display
from fastapi.testclient import TestClient

cwd = Path.cwd()
PROJECT_DIR = cwd if (cwd / "api.py").exists() else cwd.parent
if not (PROJECT_DIR / "api.py").exists():
    raise FileNotFoundError("Could not find api.py. Place this notebook inside the tests folder.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from api import app
client = TestClient(app)
API_RESULTS = []
API_STATE = {}

def record_api(test_id, description, response, expected_status, extra_pass=True, details=""):
    API_RESULTS[:] = [item for item in API_RESULTS if item["test"] != test_id]
    actual_status = response.status_code
    passed = actual_status == expected_status and bool(extra_pass)
    status = "PASS" if passed else "FAIL"
    try:
        body = response.json()
    except Exception:
        body = response.text[:500]
    API_RESULTS.append({"test": test_id, "description": description, "expected_status": expected_status, "actual_status": actual_status, "status": status, "details": details})
    print(f"{status}: {test_id} — {description}")
    print("Expected HTTP status:", expected_status)
    print("Actual HTTP status:  ", actual_status)
    print("Response:", body if len(str(body)) < 1200 else str(body)[:1200] + "...")
    if details: print("Check:", details)
    return passed

print("Setup complete")
print("Python:", sys.executable)
print("Project directory:", PROJECT_DIR)
print("FastAPI TestClient ready — Uvicorn is not required")

Setup complete
Python: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\.venv\Scripts\python.exe
Project directory: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C
FastAPI TestClient ready — Uvicorn is not required


## API Test 1 — Health endpoint

In [29]:
response = client.get("/health")
body = response.json() if response.status_code == 200 else {}
record_api("API-01", "GET /health returns healthy status", response, 200, body.get("status") == "healthy", "status must equal healthy")

PASS: API-01 — GET /health returns healthy status
Expected HTTP status: 200
Actual HTTP status:   200
Response: {'status': 'healthy', 'version': '4.0.0'}
Check: status must equal healthy


True

## API Test 2 — Dashboard endpoint

In [30]:
response = client.get("/")
content_type = response.headers.get("content-type", "")
record_api("API-02", "GET / returns dashboard", response, 200, "text/html" in content_type or "application/json" in content_type, f"content-type={content_type}")

PASS: API-02 — GET / returns dashboard
Expected HTTP status: 200
Actual HTTP status:   200
Response: <!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Erlang C Forecast & Agent Scheduling Dashboard</title>

    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background: #f4f6f8;
            colo
Check: content-type=text/html; charset=utf-8


True

## API Test 3 — Reject unsupported Excel upload

In [31]:
response = client.post("/api/v1/cdr/stl-forecast", files=[("files", ("sample.xlsx", b"fake", "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"))])
detail = response.json().get("detail", "")
record_api("API-03", "Excel upload is rejected", response, 400, "Unsupported file type" in str(detail), "error must mention unsupported file type")

PASS: API-03 — Excel upload is rejected
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': 'Unsupported file type for sample.xlsx.'}
Check: error must mention unsupported file type


True

## API Test 4 — Reject an empty CSV

In [32]:
response = client.post("/api/v1/cdr/stl-forecast", files=[("files", ("empty.csv", b"", "text/csv"))])
detail = response.json().get("detail", "")
record_api("API-04", "Empty CSV is rejected", response, 400, "empty" in str(detail).lower(), "error must mention empty file")

PASS: API-04 — Empty CSV is rejected
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': 'empty.csv is empty.'}
Check: error must mention empty file


True

## API Test 5 — Reject more than ten files

In [33]:
files = [("files", (f"file_{number}.csv", b"sample", "text/csv")) for number in range(11)]
response = client.post("/api/v1/cdr/stl-forecast", files=files)
detail = response.json().get("detail", "")
record_api("API-05", "Eleven uploaded files are rejected", response, 400, "10" in str(detail), "maximum is 10 files")

PASS: API-05 — Eleven uploaded files are rejected
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': 'Upload no more than 10 files at a time.'}
Check: maximum is 10 files


True

## API Test 6 — Reject zero forecast days

In [34]:
response = client.post("/api/v1/cdr/stl-forecast", data={"forecast_days": "0"}, files=[("files", ("sample.csv", b"sample", "text/csv"))])
record_api("API-06", "forecast_days=0 is rejected", response, 400, "forecast_days" in str(response.json().get("detail", "")), "allowed range begins at 1")

PASS: API-06 — forecast_days=0 is rejected
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': 'forecast_days must be between 1 and 3650.'}
Check: allowed range begins at 1


True

## API Test 7 — Reject more than 3,650 forecast days

In [35]:
response = client.post("/api/v1/cdr/stl-forecast", data={"forecast_days": "3651"}, files=[("files", ("sample.csv", b"sample", "text/csv"))])
record_api("API-07", "forecast_days=3651 is rejected", response, 400, "forecast_days" in str(response.json().get("detail", "")), "maximum is 3650")

PASS: API-07 — forecast_days=3651 is rejected
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': 'forecast_days must be between 1 and 3650.'}
Check: maximum is 3650


True

## API Test 8 — Reject zero maximum agents

In [36]:
response = client.post("/api/v1/cdr/stl-forecast", data={"max_agents": "0"}, files=[("files", ("sample.csv", b"sample", "text/csv"))])
record_api("API-08", "max_agents=0 is rejected", response, 400, "max_agents" in str(response.json().get("detail", "")), "allowed range begins at 1")

PASS: API-08 — max_agents=0 is rejected
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': 'max_agents must be between 1 and 10000.'}
Check: allowed range begins at 1


True

## API Test 9 — Reject more than 10,000 maximum agents

In [37]:
response = client.post("/api/v1/cdr/stl-forecast", data={"max_agents": "10001"}, files=[("files", ("sample.csv", b"sample", "text/csv"))])
record_api("API-09", "max_agents=10001 is rejected", response, 400, "max_agents" in str(response.json().get("detail", "")), "maximum is 10000")

PASS: API-09 — max_agents=10001 is rejected
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': 'max_agents must be between 1 and 10000.'}
Check: maximum is 10000


True

## API Test 10 — Upload a valid generated CDR and create a forecast

This cell creates a small temporary 2024 CSV in memory. It does not use or modify your real datasets. It may take longer than the earlier tests.

In [38]:
rows = []
for number, timestamp in enumerate(pd.date_range("2024-12-01 00:00:00", "2024-12-21 23:30:00", freq="30min"), start=1):
    rows.append(["1001", "2001", timestamp.strftime("%Y-%b-%d %I:%M:%S %p"), "00:05:00", "Answered", f"test-{number}", "0770000000"])
csv_bytes = pd.DataFrame(rows).to_csv(index=False, header=False).encode("utf-8")
response = client.post(
    "/api/v1/cdr/stl-forecast",
    data={"interval_minutes": "30", "forecast_days": "2", "target_seconds": "20", "target_service_level": "80", "shrinkage": "30", "max_agents": "1000", "include_forecast_rows": "true"},
    files=[("files", ("generated_2024.csv", csv_bytes, "text/csv"))],
)
body = response.json() if response.status_code == 200 else {}
extra_pass = body.get("output_year") == 2025 and body.get("forecast_interval_count") == 96 and len(body.get("forecast", [])) == 96
if response.status_code == 200: API_STATE["forecast_response"] = body
record_api("API-10", "Valid CDR creates a two-day forecast", response, 200, extra_pass, "output_year=2025 and 96 half-hour forecast rows")

PASS: API-10 — Valid CDR creates a two-day forecast
Expected HTTP status: 200
Actual HTTP status:   200
Response: {'method': 'STL', 'logic': 'STL weekly decomposition with linear trend extrapolation and repeated seasonal cycle.', 'dataset_count': 1, 'historical_years': [2024], 'output_year': 2025, 'days': 2, 'interval_minutes': 30, 'seasonal_period': 336, 'trend_lookback_days': 90, 'forecast_interval_count': 96, 'total_predicted_calls': 96, 'average_aht_seconds': 300.0, 'maximum_raw_agents': 1, 'maximum_scheduled_agents': 2, 'average_service_level_percent': 84.23, 'average_occupancy_percent': 16.67, 'average_asa_seconds': 60.0, 'intervals_below_service_target': 0, 'peak_interval': {'interval_start': '2025-01-01T00:00:00', 'call_volume': 1, 'scheduled_agents': 2}, 'source_data': {'dataset_count': 1, 'historical_years': [2024], 'datasets': [{'filename': 'generated_2024.csv', 'year': 2024, 'raw_rows': 1008, 'valid_answered_calls': 1008, 'removed_rows': 0, 'average_aht_seconds': 300.0, 'fi

True

## API Test 11 — Create a valid monthly schedule

In [39]:
simple_forecast = [
    {"interval_start": "2025-01-01T00:00:00", "scheduled_agents": 2},
    {"interval_start": "2025-01-01T08:00:00", "scheduled_agents": 2},
    {"interval_start": "2025-01-01T16:00:00", "scheduled_agents": 2},
]
response = client.post("/api/v1/schedule/monthly", json={"forecast": simple_forecast, "year": 2025, "month": 1, "agent_count": 6})
body = response.json() if response.status_code == 200 else {}
extra_pass = "summary" in body and "schedule" in body and body.get("summary", {}).get("coverage_ok") is True
if response.status_code == 200: API_STATE["schedule_response"] = body
record_api("API-11", "Valid monthly schedule request succeeds", response, 200, extra_pass, "response includes summary, schedule and full coverage")

PASS: API-11 — Valid monthly schedule request succeeds
Expected HTTP status: 200
Actual HTTP status:   200
Response: {'summary': {'year': 2025, 'month': 1, 'minimum_agents': 6, 'agent_count': 6, 'shift_hours': 8, 'working_days_per_week': 5, 'days_off_per_week': 2, 'total_required_shift_assignments': 6, 'total_assigned_shift_assignments': 6, 'coverage_shortage': 0, 'coverage_ok': True, 'coverage': [{'date': '2025-01-01', 'weekday': 'Wednesday', 'shift_code': 'NIGHT', 'shift': '00:00-08:00', 'required_agents': 2, 'assigned_agents': 2, 'shortage': 0, 'status': 'OK'}, {'date': '2025-01-01', 'weekday': 'Wednesday', 'shift_code': 'MORNING', 'shift': '08:00-16:00', 'required_agents': 2, 'assigned_agents': 2, 'shortage': 0, 'status': 'OK'}, {'date': '2025-01-01', 'weekday': 'Wednesday', 'shift_code': 'EVENING', 'shift': '16:00-00:00', 'required_agents': 2, 'assigned_agents': 2, 'shortage': 0, 'status': 'OK'}]}, 'schedule': [{'agent_id': 'Agent 001', 'date': '2025-01-01', 'weekday': 'Wednesday'

True

## API Test 12 — Reject missing forecast columns

In [40]:
response = client.post("/api/v1/schedule/monthly", json={"forecast": [{"interval_start": "2025-01-01T00:00:00"}], "year": 2025, "month": 1})
detail = response.json().get("detail", "")
record_api("API-12", "Missing scheduled_agents is rejected", response, 400, "scheduled_agents" in str(detail), "error must identify the missing column")

PASS: API-12 — Missing scheduled_agents is rejected
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': "Forecast is missing required columns: ['scheduled_agents']"}
Check: error must identify the missing column


True

## API Test 13 — Reject invalid month

In [41]:
response = client.post("/api/v1/schedule/monthly", json={"forecast": simple_forecast, "year": 2025, "month": 13})
record_api("API-13", "Month 13 is rejected by request validation", response, 422, True, "FastAPI/Pydantic validation error expected")

PASS: API-13 — Month 13 is rejected by request validation
Expected HTTP status: 422
Actual HTTP status:   422
Response: {'detail': [{'type': 'less_than_equal', 'loc': ['body', 'month'], 'msg': 'Input should be less than or equal to 12', 'input': 13, 'ctx': {'le': 12}}]}
Check: FastAPI/Pydantic validation error expected


True

## API Test 14 — Reject more than 10,000 agents

In [42]:
response = client.post("/api/v1/schedule/monthly", json={"forecast": simple_forecast, "year": 2025, "month": 1, "agent_count": 10001})
record_api("API-14", "Agent count 10001 is rejected", response, 422, True, "FastAPI/Pydantic validation error expected")

PASS: API-14 — Agent count 10001 is rejected
Expected HTTP status: 422
Actual HTTP status:   422
Response: {'detail': [{'type': 'less_than_equal', 'loc': ['body', 'agent_count'], 'msg': 'Input should be less than or equal to 10000', 'input': 10001, 'ctx': {'le': 10000}}]}
Check: FastAPI/Pydantic validation error expected


True

## API Test 15 — Reject incomplete leave schedule data

In [43]:
response = client.post("/api/v1/schedule/leave", json={
    "forecast": simple_forecast,
    "schedule": [{"agent_id": "Agent 001", "date": "2025-01-01"}],
    "agent_id": "Agent 001",
    "leave_date": "2025-01-01",
})
detail = response.json().get("detail", "")
record_api("API-15", "Incomplete schedule columns are rejected for leave", response, 400, "missing required columns" in str(detail).lower(), "shift_code, shift and status are missing")

PASS: API-15 — Incomplete schedule columns are rejected for leave
Expected HTTP status: 400
Actual HTTP status:   400
Response: {'detail': "Schedule is missing required columns: ['shift', 'shift_code', 'status']"}
Check: shift_code, shift and status are missing


True

## Final API test report — run after API Tests 1–15

In [44]:
expected_ids = [f"API-{number:02d}" for number in range(1, 16)]
completed = {item["test"] for item in API_RESULTS}
not_run = [test_id for test_id in expected_ids if test_id not in completed]
passed = sum(item["status"] == "PASS" for item in API_RESULTS)
failed = sum(item["status"] == "FAIL" for item in API_RESULTS)
result_rows = []
for test_id in expected_ids:
    item = next((row for row in API_RESULTS if row["test"] == test_id), None)
    if item:
        result_rows.append(f"| {item['test']} | {item['description']} | {item['expected_status']} | {item['actual_status']} | {item['status']} |")
    else:
        result_rows.append(f"| {test_id} | Not run | — | — | NOT RUN |")
final_status = "PASS" if failed == 0 and not not_run else ("FAIL" if failed else "INCOMPLETE")
report = f"""
# Erlang C API Testing Report

## Test information

- Test date: {datetime.now().strftime('%Y-%m-%d %H:%M')}
- Application: FastAPI Erlang C Calculator
- Test method: FastAPI TestClient
- External server required: No

## Overall results

| Result | Count |
|---|---:|
| Expected tests | {len(expected_ids)} |
| Passed | {passed} |
| Failed | {failed} |
| Not run | {len(not_run)} |

## Detailed results

| Test | Description | Expected HTTP status | Actual HTTP status | Result |
|---|---|---:|---:|---|
{chr(10).join(result_rows)}

## Final status

{final_status}

{('All 15 API tests passed.' if final_status == 'PASS' else ('One or more API tests failed and should be reviewed.' if final_status == 'FAIL' else 'Run all API test cells before creating the final result.'))}
"""
display(Markdown(report))


# Erlang C API Testing Report

## Test information

- Test date: 2026-09-04 22:37
- Application: FastAPI Erlang C Calculator
- Test method: FastAPI TestClient
- External server required: No

## Overall results

| Result | Count |
|---|---:|
| Expected tests | 15 |
| Passed | 15 |
| Failed | 0 |
| Not run | 0 |

## Detailed results

| Test | Description | Expected HTTP status | Actual HTTP status | Result |
|---|---|---:|---:|---|
| API-01 | GET /health returns healthy status | 200 | 200 | PASS |
| API-02 | GET / returns dashboard | 200 | 200 | PASS |
| API-03 | Excel upload is rejected | 400 | 400 | PASS |
| API-04 | Empty CSV is rejected | 400 | 400 | PASS |
| API-05 | Eleven uploaded files are rejected | 400 | 400 | PASS |
| API-06 | forecast_days=0 is rejected | 400 | 400 | PASS |
| API-07 | forecast_days=3651 is rejected | 400 | 400 | PASS |
| API-08 | max_agents=0 is rejected | 400 | 400 | PASS |
| API-09 | max_agents=10001 is rejected | 400 | 400 | PASS |
| API-10 | Valid CDR creates a two-day forecast | 200 | 200 | PASS |
| API-11 | Valid monthly schedule request succeeds | 200 | 200 | PASS |
| API-12 | Missing scheduled_agents is rejected | 400 | 400 | PASS |
| API-13 | Month 13 is rejected by request validation | 422 | 422 | PASS |
| API-14 | Agent count 10001 is rejected | 422 | 422 | PASS |
| API-15 | Incomplete schedule columns are rejected for leave | 400 | 400 | PASS |

## Final status

PASS

All 15 API tests passed.
